In [3]:
import csv
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

with open("wine_dataset.csv", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    features = [name for name in reader.fieldnames if name != "target"]

X = torch.tensor([[float(row[c]) for c in features] for row in rows], dtype=torch.float32)
y = torch.tensor([int(row["target"]) for row in rows], dtype=torch.long)

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("classes:", torch.unique(y, return_counts=True))
assert X.shape == (178, 13) and y.shape == (178,)

X: torch.Size([178, 13]) torch.float32
y: torch.Size([178]) torch.int64
classes: (tensor([0, 1, 2]), tensor([59, 71, 48]))


In [5]:
def stratified_split(y, test_fraction=0.2, seed=42):
    g = torch.Generator().manual_seed(seed)
    train_parts, test_parts = [], []
    for cls in torch.unique(y):
        indices = torch.where(y == cls)[0]
        indices = indices[torch.randperm(indices.numel(), generator=g)]
        n_test = round(indices.numel() * test_fraction)
        test_parts.append(indices[:n_test])
        train_parts.append(indices[n_test:])
    return torch.cat(train_parts), torch.cat(test_parts)

train_idx, test_idx = stratified_split(y)
assert len(train_idx) + len(test_idx) == len(y)
assert not set(train_idx.tolist()) & set(test_idx.tolist())
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print("train/test:", len(train_idx), len(test_idx))
print("train class distribution:", torch.bincount(y_train))
print("test class distribution:", torch.bincount(y_test))

train/test: 142 36
train class distribution: tensor([47, 57, 38])
test class distribution: tensor([12, 14, 10])


In [ ]:
def standardize(x_train, x_other):
    # TODO: вернуть (normalized_train, normalized_other)
    raise NotImplementedError("Реализуй standardize")

xtr, xte = standardize(X_train, X_test)
assert xtr.shape == X_train.shape and xte.shape == X_test.shape
assert torch.all(torch.isfinite(xtr)) and torch.all(torch.isfinite(xte))
assert torch.allclose(xtr.mean(0), torch.zeros(13), atol=1e-5)
assert torch.allclose(xtr.std(0, unbiased=False), torch.ones(13), atol=1e-5)

In [ ]:
def dummy_accuracy(y_train, y_other):
    # TODO: float в пределах [0,1]
    raise NotImplementedError("Реализуй dummy_accuracy")

print("Dummy train accuracy (только самопроверка):",
      dummy_accuracy(y_train, y_train))

In [ ]:
torch.manual_seed(42)
# TODO: замени строку ниже своей моделью
model = None

# assert model(torch.randn(5, 13)).shape == (5, 3)

In [ ]:
def fit(train_x, train_y, epochs=250, seed=42):
    # TODO: создать новую модель при КАЖДОМ вызове (важно для CV!)
    # TODO: DataLoader, loss, optimizer, train loop
    # TODO: return model
    raise NotImplementedError("Реализуй fit")

In [ ]:
@torch.no_grad()
def accuracy(model, x, y):
    # TODO: model.eval(); logits -> predicted class -> mean(correct)
    raise NotImplementedError("Реализуй accuracy")

In [ ]:
def make_stratified_folds(y, k=5, seed=42):
    # TODO
    raise NotImplementedError("Реализуй make_stratified_folds")

folds = make_stratified_folds(y_train)
all_valid = torch.cat([val for _, val in folds])
assert len(folds) == 5
assert sorted(all_valid.tolist()) == list(range(len(y_train)))
for tr, val in folds:
    assert not set(tr.tolist()) & set(val.tolist())

In [ ]:
cv_scores = []
dummy_scores = []
for fold_number, (tr_idx, val_idx) in enumerate(folds, start=1):
    xtr, xval = standardize(X_train[tr_idx], X_train[val_idx])
    model = fit(xtr, y_train[tr_idx])
    score = accuracy(model, xval, y_train[val_idx])
    baseline = dummy_accuracy(y_train[tr_idx], y_train[val_idx])
    cv_scores.append(score)
    dummy_scores.append(baseline)
    print(f"fold {fold_number}: torch={score:.3f}, dummy={baseline:.3f}")

scores = torch.tensor(cv_scores)
print("CV mean:", scores.mean().item())
print("CV std:", scores.std(unbiased=False).item())
print("dummy mean:", torch.tensor(dummy_scores).mean().item())

In [ ]:
xtr, xte = standardize(X_train, X_test)
final_model = fit(xtr, y_train)
print("Final test accuracy:", accuracy(final_model, xte, y_test))